# ME2N — Pedidos de Compra (tabela base)

**Tabela:** `dev_procurement.corp_curated.tbl_ds_proc_me2n`
**Transação SAP:** ME2N · **Colunas:** 47
**Clustering declarado:** `num_pedido_compra`, `cod_material`

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros criados via notebook UI (SQL warehouse nao suporta CREATE WIDGET).
-- Referenciados nas demais celulas via :f_cod_centro, :f_cod_empresa, etc.
SELECT
  :f_cod_centro AS filtro_centro,
  :f_cod_empresa AS filtro_empresa,
  :f_cod_organizacao_compras AS filtro_org_compras,
  :f_cod_grupo_compradores AS filtro_grupo_compradores;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
WHERE (:f_cod_centro = '' OR `cod_centro` = :f_cod_centro)
  AND (:f_cod_empresa = '' OR `cod_empresa` = :f_cod_empresa)
  AND (:f_cod_organizacao_compras = '' OR `cod_organizacao_compras` = :f_cod_organizacao_compras)
  AND (:f_cod_grupo_compradores = '' OR `cod_grupo_compradores` = :f_cod_grupo_compradores);
-- Para teste rapido, descomente:
-- LIMIT 1000000;

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_proc_me2n;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_proc_me2n;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_proc_me2n LIMIT 20;

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM base)
UNION ALL
SELECT 'num_pedido_compra + num_item_pedido_compra + cod_material' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `cod_material` FROM base)
UNION ALL
SELECT 'num_pedido_compra' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_pedido_compra` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_pedido_compra`, `num_item_pedido_compra`, COUNT(*) AS qtd FROM base GROUP BY `num_pedido_compra`, `num_item_pedido_compra` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'num_pedido_compra + num_item_pedido_compra + cod_material' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_pedido_compra`, `num_item_pedido_compra`, `cod_material`, COUNT(*) AS qtd FROM base GROUP BY `num_pedido_compra`, `num_item_pedido_compra`, `cod_material` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'num_pedido_compra' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_pedido_compra`, COUNT(*) AS qtd FROM base GROUP BY `num_pedido_compra` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(47,
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_texto_breve', 'string', COUNT_IF(`cod_texto_breve` IS NULL), COUNT_IF(`cod_texto_breve` IS NOT NULL AND lower(trim(`cod_texto_breve`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_texto_breve`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mercadorias', 'string', COUNT_IF(`tp_grupo_mercadorias` IS NULL), COUNT_IF(`tp_grupo_mercadorias` IS NOT NULL AND lower(trim(`tp_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'cod_unidade_medida_pedido', 'string', COUNT_IF(`cod_unidade_medida_pedido` IS NULL), COUNT_IF(`cod_unidade_medida_pedido` IS NOT NULL AND lower(trim(`cod_unidade_medida_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida_pedido`) RLIKE '^0+([.,]0+)?$'),
    'cod_unidade_medida_basica', 'string', COUNT_IF(`cod_unidade_medida_basica` IS NULL), COUNT_IF(`cod_unidade_medida_basica` IS NOT NULL AND lower(trim(`cod_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'vl_preco_liquido', 'double', COUNT_IF(`vl_preco_liquido` IS NULL), 0L, COUNT_IF(`vl_preco_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'st_liberacao', 'string', COUNT_IF(`st_liberacao` IS NULL), COUNT_IF(`st_liberacao` IS NOT NULL AND lower(trim(`st_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_liberacao', 'string', COUNT_IF(`cod_liberacao` IS NULL), COUNT_IF(`cod_liberacao` IS NOT NULL AND lower(trim(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_estrat_liberacao', 'string', COUNT_IF(`cod_estrat_liberacao` IS NULL), COUNT_IF(`cod_estrat_liberacao` IS NOT NULL AND lower(trim(`cod_estrat_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estrat_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_organizacao_compras', 'string', COUNT_IF(`cod_organizacao_compras` IS NULL), COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND lower(trim(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'tp_classificacao_contabil', 'string', COUNT_IF(`tp_classificacao_contabil` IS NULL), COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND lower(trim(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_documento_compras', 'string', COUNT_IF(`tp_documento_compras` IS NULL), COUNT_IF(`tp_documento_compras` IS NOT NULL AND lower(trim(`tp_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_documento', 'string', COUNT_IF(`tp_categoria_documento` IS NULL), COUNT_IF(`tp_categoria_documento` IS NOT NULL AND lower(trim(`tp_categoria_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_documento`) RLIKE '^0+([.,]0+)?$'),
    'cod_categoria_item', 'string', COUNT_IF(`cod_categoria_item` IS NULL), COUNT_IF(`cod_categoria_item` IS NOT NULL AND lower(trim(`cod_categoria_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_categoria_item`) RLIKE '^0+([.,]0+)?$'),
    'desc_categoria_item', 'string', COUNT_IF(`desc_categoria_item` IS NULL), COUNT_IF(`desc_categoria_item` IS NOT NULL AND lower(trim(`desc_categoria_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_categoria_item`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_fornecedor', 'string', COUNT_IF(`cod_centro_fornecedor` IS NULL), COUNT_IF(`cod_centro_fornecedor` IS NOT NULL AND lower(trim(`cod_centro_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'num_contrato_basico', 'string', COUNT_IF(`num_contrato_basico` IS NULL), COUNT_IF(`num_contrato_basico` IS NOT NULL AND lower(trim(`num_contrato_basico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_contrato_basico`) RLIKE '^0+([.,]0+)?$'),
    'num_registro_info', 'string', COUNT_IF(`num_registro_info` IS NULL), COUNT_IF(`num_registro_info` IS NOT NULL AND lower(trim(`num_registro_info`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_registro_info`) RLIKE '^0+([.,]0+)?$'),
    'cod_imposto', 'string', COUNT_IF(`cod_imposto` IS NULL), COUNT_IF(`cod_imposto` IS NOT NULL AND lower(trim(`cod_imposto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_imposto`) RLIKE '^0+([.,]0+)?$'),
    'ind_entrega_concluida', 'string', COUNT_IF(`ind_entrega_concluida` IS NULL), COUNT_IF(`ind_entrega_concluida` IS NOT NULL AND lower(trim(`ind_entrega_concluida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_entrega_concluida`) RLIKE '^0+([.,]0+)?$'),
    'num_acompanhamento', 'string', COUNT_IF(`num_acompanhamento` IS NULL), COUNT_IF(`num_acompanhamento` IS NOT NULL AND lower(trim(`num_acompanhamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_acompanhamento`) RLIKE '^0+([.,]0+)?$'),
    'dt_pedido', 'string', COUNT_IF(`dt_pedido` IS NULL), COUNT_IF(`dt_pedido` IS NOT NULL AND lower(trim(`dt_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_pedido`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa_item', 'string', COUNT_IF(`dt_remessa_item` IS NULL), COUNT_IF(`dt_remessa_item` IS NOT NULL AND lower(trim(`dt_remessa_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_remessa_item`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa_primeira', 'string', COUNT_IF(`dt_remessa_primeira` IS NULL), COUNT_IF(`dt_remessa_primeira` IS NOT NULL AND lower(trim(`dt_remessa_primeira`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_remessa_primeira`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa_ultima', 'string', COUNT_IF(`dt_remessa_ultima` IS NULL), COUNT_IF(`dt_remessa_ultima` IS NOT NULL AND lower(trim(`dt_remessa_ultima`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_remessa_ultima`) RLIKE '^0+([.,]0+)?$'),
    'qt_divisoes_remessa', 'bigint', COUNT_IF(`qt_divisoes_remessa` IS NULL), 0L, COUNT_IF(`qt_divisoes_remessa` = 0),
    'cod_requisicao_compra', 'string', COUNT_IF(`cod_requisicao_compra` IS NULL), COUNT_IF(`cod_requisicao_compra` IS NOT NULL AND lower(trim(`cod_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_item_requisicao_compra', 'string', COUNT_IF(`cod_item_requisicao_compra` IS NULL), COUNT_IF(`cod_item_requisicao_compra` IS NOT NULL AND lower(trim(`cod_item_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_item_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'vl_requisicao_compra', 'double', COUNT_IF(`vl_requisicao_compra` IS NULL), 0L, COUNT_IF(`vl_requisicao_compra` = 0),
    'qt_unidade_preco_requisicao', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco_requisicao` IS NULL), 0L, COUNT_IF(`qt_unidade_preco_requisicao` = 0),
    'qt_solicitada', 'decimal(13,3)', COUNT_IF(`qt_solicitada` IS NULL), 0L, COUNT_IF(`qt_solicitada` = 0),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(47,
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'cod_texto_breve', 'string', approx_count_distinct(`cod_texto_breve`),
    'tp_grupo_mercadorias', 'string', approx_count_distinct(`tp_grupo_mercadorias`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'cod_unidade_medida_pedido', 'string', approx_count_distinct(`cod_unidade_medida_pedido`),
    'cod_unidade_medida_basica', 'string', approx_count_distinct(`cod_unidade_medida_basica`),
    'vl_preco_liquido', 'double', approx_count_distinct(`vl_preco_liquido`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'st_liberacao', 'string', approx_count_distinct(`st_liberacao`),
    'cod_liberacao', 'string', approx_count_distinct(`cod_liberacao`),
    'cod_estrat_liberacao', 'string', approx_count_distinct(`cod_estrat_liberacao`),
    'cod_organizacao_compras', 'string', approx_count_distinct(`cod_organizacao_compras`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'tp_classificacao_contabil', 'string', approx_count_distinct(`tp_classificacao_contabil`),
    'tp_documento_compras', 'string', approx_count_distinct(`tp_documento_compras`),
    'tp_categoria_documento', 'string', approx_count_distinct(`tp_categoria_documento`),
    'cod_categoria_item', 'string', approx_count_distinct(`cod_categoria_item`),
    'desc_categoria_item', 'string', approx_count_distinct(`desc_categoria_item`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'cod_centro_fornecedor', 'string', approx_count_distinct(`cod_centro_fornecedor`),
    'num_contrato_basico', 'string', approx_count_distinct(`num_contrato_basico`),
    'num_registro_info', 'string', approx_count_distinct(`num_registro_info`),
    'cod_imposto', 'string', approx_count_distinct(`cod_imposto`),
    'ind_entrega_concluida', 'string', approx_count_distinct(`ind_entrega_concluida`),
    'num_acompanhamento', 'string', approx_count_distinct(`num_acompanhamento`),
    'dt_pedido', 'string', approx_count_distinct(`dt_pedido`),
    'dt_criacao', 'string', approx_count_distinct(`dt_criacao`),
    'dt_remessa_item', 'string', approx_count_distinct(`dt_remessa_item`),
    'dt_remessa_primeira', 'string', approx_count_distinct(`dt_remessa_primeira`),
    'dt_remessa_ultima', 'string', approx_count_distinct(`dt_remessa_ultima`),
    'qt_divisoes_remessa', 'bigint', approx_count_distinct(`qt_divisoes_remessa`),
    'cod_requisicao_compra', 'string', approx_count_distinct(`cod_requisicao_compra`),
    'cod_item_requisicao_compra', 'string', approx_count_distinct(`cod_item_requisicao_compra`),
    'vl_requisicao_compra', 'double', approx_count_distinct(`vl_requisicao_compra`),
    'qt_unidade_preco_requisicao', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco_requisicao`),
    'qt_solicitada', 'decimal(13,3)', approx_count_distinct(`qt_solicitada`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'tp_grupo_mercadorias' AS coluna, CAST(`tp_grupo_mercadorias` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_liberacao' AS coluna, CAST(`st_liberacao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `st_liberacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_organizacao_compras' AS coluna, CAST(`cod_organizacao_compras` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_organizacao_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_classificacao_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_documento_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_categoria_documento' AS coluna, CAST(`tp_categoria_documento` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_categoria_documento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_categoria_item' AS coluna, CAST(`cod_categoria_item` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_categoria_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_categoria_item' AS coluna, CAST(`desc_categoria_item` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `desc_categoria_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_entrega_concluida' AS coluna, CAST(`ind_entrega_concluida` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_entrega_concluida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(6,
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.95) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0),
    'vl_preco_liquido', 'double', COUNT(`vl_preco_liquido`), CAST(MIN(`vl_preco_liquido`) AS DOUBLE), CAST(MAX(`vl_preco_liquido`) AS DOUBLE), CAST(AVG(`vl_preco_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_preco_liquido`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_preco_liquido`, 0.95) AS DOUBLE), COUNT_IF(`vl_preco_liquido` < 0), COUNT_IF(`vl_preco_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.95) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0),
    'qt_divisoes_remessa', 'bigint', COUNT(`qt_divisoes_remessa`), CAST(MIN(`qt_divisoes_remessa`) AS DOUBLE), CAST(MAX(`qt_divisoes_remessa`) AS DOUBLE), CAST(AVG(`qt_divisoes_remessa`) AS DOUBLE), CAST(percentile_approx(`qt_divisoes_remessa`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_divisoes_remessa`, 0.95) AS DOUBLE), COUNT_IF(`qt_divisoes_remessa` < 0), COUNT_IF(`qt_divisoes_remessa` = 0),
    'vl_requisicao_compra', 'double', COUNT(`vl_requisicao_compra`), CAST(MIN(`vl_requisicao_compra`) AS DOUBLE), CAST(MAX(`vl_requisicao_compra`) AS DOUBLE), CAST(AVG(`vl_requisicao_compra`) AS DOUBLE), CAST(percentile_approx(`vl_requisicao_compra`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_requisicao_compra`, 0.95) AS DOUBLE), COUNT_IF(`vl_requisicao_compra` < 0), COUNT_IF(`vl_requisicao_compra` = 0),
    'qt_solicitada', 'decimal(13,3)', COUNT(`qt_solicitada`), CAST(MIN(`qt_solicitada`) AS DOUBLE), CAST(MAX(`qt_solicitada`) AS DOUBLE), CAST(AVG(`qt_solicitada`) AS DOUBLE), CAST(percentile_approx(`qt_solicitada`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_solicitada`, 0.95) AS DOUBLE), COUNT_IF(`qt_solicitada` < 0), COUNT_IF(`qt_solicitada` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10. Datas armazenadas como STRING

**Armadilha conhecida:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
É a mesma data em formato diferente — já gerou **16.773 falsos positivos**.

Se a coluna `veredito` acusar mais de um formato, a normalização é obrigatória.

In [0]:
-- 10. DATAS ARMAZENADAS COMO STRING
-- ARMADILHA: SAP exporta '2024-02-23 00:00:00', Datalake grava '20240223'.
-- Mesma data, formato diferente. Ja gerou 16.773 falsos positivos.
WITH t AS (SELECT COUNT(*) AS total FROM base),
d AS (
  SELECT stack(5,
    'dt_pedido', COUNT_IF(`dt_pedido` IS NULL OR trim(`dt_pedido`) = ''), COUNT_IF(trim(`dt_pedido`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_pedido`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_pedido`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_pedido`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_pedido`) NOT IN ('', '00000000') THEN `dt_pedido` END), MAX(CASE WHEN trim(`dt_pedido`) NOT IN ('', '00000000') THEN `dt_pedido` END),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_criacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END),
    'dt_remessa_item', COUNT_IF(`dt_remessa_item` IS NULL OR trim(`dt_remessa_item`) = ''), COUNT_IF(trim(`dt_remessa_item`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_remessa_item`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_remessa_item`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_remessa_item`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_remessa_item`) NOT IN ('', '00000000') THEN `dt_remessa_item` END), MAX(CASE WHEN trim(`dt_remessa_item`) NOT IN ('', '00000000') THEN `dt_remessa_item` END),
    'dt_remessa_primeira', COUNT_IF(`dt_remessa_primeira` IS NULL OR trim(`dt_remessa_primeira`) = ''), COUNT_IF(trim(`dt_remessa_primeira`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_remessa_primeira`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_remessa_primeira`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_remessa_primeira`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_remessa_primeira`) NOT IN ('', '00000000') THEN `dt_remessa_primeira` END), MAX(CASE WHEN trim(`dt_remessa_primeira`) NOT IN ('', '00000000') THEN `dt_remessa_primeira` END),
    'dt_remessa_ultima', COUNT_IF(`dt_remessa_ultima` IS NULL OR trim(`dt_remessa_ultima`) = ''), COUNT_IF(trim(`dt_remessa_ultima`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_remessa_ultima`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_remessa_ultima`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_remessa_ultima`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_remessa_ultima`) NOT IN ('', '00000000') THEN `dt_remessa_ultima` END), MAX(CASE WHEN trim(`dt_remessa_ultima`) NOT IN ('', '00000000') THEN `dt_remessa_ultima` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM base
)
SELECT d.coluna, d.vazios, d.fmt_AAAAMMDD, d.fmt_ISO, d.fmt_BR, d.data_zero,
       t.total - d.vazios - d.fmt_AAAAMMDD - d.fmt_ISO - d.fmt_BR AS nao_reconhecido,
       d.minimo, d.maximo,
       CASE WHEN (CASE WHEN d.fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato na mesma coluna'
            ELSE 'formato unico' END AS veredito
FROM d CROSS JOIN t
ORDER BY d.coluna;

## 10.1 Datas em tipo nativo

In [0]:
-- 10.1 DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(1,
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date()), COUNT_IF(`dateingest` < DATE'1990-01-01')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, anteriores_1990)
  FROM base
)
ORDER BY coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(8,
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_pedido_compra` AS STRING) <> trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) <> trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_fornecedor` AS STRING) <> trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'num_contrato_basico', 'string', COUNT_IF(CAST(`num_contrato_basico` AS STRING) IS NULL OR trim(CAST(`num_contrato_basico` AS STRING)) = ''), MIN(length(trim(CAST(`num_contrato_basico` AS STRING)))), MAX(length(trim(CAST(`num_contrato_basico` AS STRING)))), COUNT_IF(trim(CAST(`num_contrato_basico` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_contrato_basico` AS STRING) <> trim(CAST(`num_contrato_basico` AS STRING))), COUNT(DISTINCT trim(CAST(`num_contrato_basico` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_contrato_basico` AS STRING)), '^0+', '')),
    'num_registro_info', 'string', COUNT_IF(CAST(`num_registro_info` AS STRING) IS NULL OR trim(CAST(`num_registro_info` AS STRING)) = ''), MIN(length(trim(CAST(`num_registro_info` AS STRING)))), MAX(length(trim(CAST(`num_registro_info` AS STRING)))), COUNT_IF(trim(CAST(`num_registro_info` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_registro_info` AS STRING) <> trim(CAST(`num_registro_info` AS STRING))), COUNT(DISTINCT trim(CAST(`num_registro_info` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_registro_info` AS STRING)), '^0+', '')),
    'cod_requisicao_compra', 'string', COUNT_IF(CAST(`cod_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`cod_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`cod_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_requisicao_compra` AS STRING) <> trim(CAST(`cod_requisicao_compra` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_requisicao_compra` AS STRING)), '^0+', '')),
    'cod_centro_fornecedor', 'string', COUNT_IF(CAST(`cod_centro_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_centro_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_centro_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro_fornecedor` AS STRING) <> trim(CAST(`cod_centro_fornecedor` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_fornecedor` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro
SELECT `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_empresa
SELECT `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_empresa`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_organizacao_compras
SELECT `cod_organizacao_compras`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_organizacao_compras`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_grupo_compradores
SELECT `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_grupo_compradores`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **num_pedido_compra + cod_material**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `num_pedido_compra`, `cod_material`, COUNT(*) AS qtd
FROM base
GROUP BY `num_pedido_compra`, `cod_material`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `num_pedido_compra`, `cod_material` FROM base GROUP BY `num_pedido_compra`, `cod_material` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`num_pedido_compra`, `cod_material`)
),
agg AS (
  SELECT `num_pedido_compra`, `cod_material`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `cod_texto_breve`) AS `cod_texto_breve`,
         COUNT(DISTINCT `tp_grupo_mercadorias`) AS `tp_grupo_mercadorias`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `cod_unidade_medida_pedido`) AS `cod_unidade_medida_pedido`,
         COUNT(DISTINCT `cod_unidade_medida_basica`) AS `cod_unidade_medida_basica`,
         COUNT(DISTINCT `vl_preco_liquido`) AS `vl_preco_liquido`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `st_liberacao`) AS `st_liberacao`,
         COUNT(DISTINCT `cod_liberacao`) AS `cod_liberacao`,
         COUNT(DISTINCT `cod_estrat_liberacao`) AS `cod_estrat_liberacao`,
         COUNT(DISTINCT `cod_organizacao_compras`) AS `cod_organizacao_compras`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `tp_classificacao_contabil`) AS `tp_classificacao_contabil`,
         COUNT(DISTINCT `tp_documento_compras`) AS `tp_documento_compras`,
         COUNT(DISTINCT `tp_categoria_documento`) AS `tp_categoria_documento`,
         COUNT(DISTINCT `cod_categoria_item`) AS `cod_categoria_item`,
         COUNT(DISTINCT `desc_categoria_item`) AS `desc_categoria_item`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `cod_centro_fornecedor`) AS `cod_centro_fornecedor`,
         COUNT(DISTINCT `num_contrato_basico`) AS `num_contrato_basico`,
         COUNT(DISTINCT `num_registro_info`) AS `num_registro_info`,
         COUNT(DISTINCT `cod_imposto`) AS `cod_imposto`,
         COUNT(DISTINCT `ind_entrega_concluida`) AS `ind_entrega_concluida`,
         COUNT(DISTINCT `num_acompanhamento`) AS `num_acompanhamento`,
         COUNT(DISTINCT `dt_pedido`) AS `dt_pedido`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `dt_remessa_item`) AS `dt_remessa_item`,
         COUNT(DISTINCT `dt_remessa_primeira`) AS `dt_remessa_primeira`,
         COUNT(DISTINCT `dt_remessa_ultima`) AS `dt_remessa_ultima`,
         COUNT(DISTINCT `qt_divisoes_remessa`) AS `qt_divisoes_remessa`,
         COUNT(DISTINCT `cod_requisicao_compra`) AS `cod_requisicao_compra`,
         COUNT(DISTINCT `cod_item_requisicao_compra`) AS `cod_item_requisicao_compra`,
         COUNT(DISTINCT `vl_requisicao_compra`) AS `vl_requisicao_compra`,
         COUNT(DISTINCT `qt_unidade_preco_requisicao`) AS `qt_unidade_preco_requisicao`,
         COUNT(DISTINCT `qt_solicitada`) AS `qt_solicitada`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `num_pedido_compra`, `cod_material`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(45,
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'desc_material', MAX(`desc_material`),
    'cod_texto_breve', MAX(`cod_texto_breve`),
    'tp_grupo_mercadorias', MAX(`tp_grupo_mercadorias`),
    'qt_pedido', MAX(`qt_pedido`),
    'cod_unidade_medida_pedido', MAX(`cod_unidade_medida_pedido`),
    'cod_unidade_medida_basica', MAX(`cod_unidade_medida_basica`),
    'vl_preco_liquido', MAX(`vl_preco_liquido`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`),
    'cod_moeda', MAX(`cod_moeda`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'st_liberacao', MAX(`st_liberacao`),
    'cod_liberacao', MAX(`cod_liberacao`),
    'cod_estrat_liberacao', MAX(`cod_estrat_liberacao`),
    'cod_organizacao_compras', MAX(`cod_organizacao_compras`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_centro', MAX(`cod_centro`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'tp_classificacao_contabil', MAX(`tp_classificacao_contabil`),
    'tp_documento_compras', MAX(`tp_documento_compras`),
    'tp_categoria_documento', MAX(`tp_categoria_documento`),
    'cod_categoria_item', MAX(`cod_categoria_item`),
    'desc_categoria_item', MAX(`desc_categoria_item`),
    'cod_deposito', MAX(`cod_deposito`),
    'cod_centro_fornecedor', MAX(`cod_centro_fornecedor`),
    'num_contrato_basico', MAX(`num_contrato_basico`),
    'num_registro_info', MAX(`num_registro_info`),
    'cod_imposto', MAX(`cod_imposto`),
    'ind_entrega_concluida', MAX(`ind_entrega_concluida`),
    'num_acompanhamento', MAX(`num_acompanhamento`),
    'dt_pedido', MAX(`dt_pedido`),
    'dt_criacao', MAX(`dt_criacao`),
    'dt_remessa_item', MAX(`dt_remessa_item`),
    'dt_remessa_primeira', MAX(`dt_remessa_primeira`),
    'dt_remessa_ultima', MAX(`dt_remessa_ultima`),
    'qt_divisoes_remessa', MAX(`qt_divisoes_remessa`),
    'cod_requisicao_compra', MAX(`cod_requisicao_compra`),
    'cod_item_requisicao_compra', MAX(`cod_item_requisicao_compra`),
    'vl_requisicao_compra', MAX(`vl_requisicao_compra`),
    'qt_unidade_preco_requisicao', MAX(`qt_unidade_preco_requisicao`),
    'qt_solicitada', MAX(`qt_solicitada`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
SELECT `dateingest` AS data_carga,
       COUNT(*) AS linhas,
       COUNT(DISTINCT num_pedido_compra) AS chaves_distintas
FROM base
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 15.1 SNAPSHOT OU HISTORICO?
SELECT MIN(`dateingest`) AS primeira_carga,
       MAX(`dateingest`) AS ultima_carga,
       COUNT(DISTINCT `dateingest`) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT `dateingest`) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir a data de carga'
       END AS veredito
FROM base;

## 16. Análises específicas — ME2N

### 16.1 Comparação tabela base × view

As duas fontes têm as mesmas 47 colunas, mas **tipos diferentes**:

| Coluna | Tabela base | View |
|---|---|---|
| `num_pedido_compra` | `string` | `bigint` |
| `cod_material` | `string` | `bigint` |
| `cod_fornecedor` | `string` | `bigint` |
| datas `dt_*` | `string` | `date` |

**Risco:** converter código para `bigint` **destrói zeros à esquerda**.

In [0]:
-- 16.1 CONTAGEM: TABELA BASE x VIEW
-- A view usa CAST(num_pedido_compra AS BIGINT) que falha para valores nao numericos (ex: '412A').
-- Para contornar, simulamos o cast da view com try_cast sobre a tabela base.
SELECT 'tbl_ds_proc_me2n (base)' AS fonte, COUNT(*) AS linhas,
       COUNT(DISTINCT num_pedido_compra) AS pedidos
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'vw_ds_proc_me2n_exibicao (view)', COUNT(*),
       COUNT(DISTINCT try_cast(num_pedido_compra AS BIGINT))
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n;

### 16.2 Medição da perda de zeros à esquerda

Consulta feita sempre sobre a **tabela base** (onde os códigos ainda são `string`).
Se houver zeros à esquerda, o cast para `bigint` da view os elimina.

In [0]:
-- 16.2 PERDA DE ZEROS A ESQUERDA NO CAST DA VIEW
SELECT 'num_pedido_compra' AS coluna,
       COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`num_pedido_compra` IS NOT NULL AND NOT trim(`num_pedido_compra`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`num_pedido_compra`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`num_pedido_compra`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'num_item_pedido_compra' AS coluna,
       COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND NOT trim(`num_item_pedido_compra`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`num_item_pedido_compra`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`num_item_pedido_compra`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'cod_material' AS coluna,
       COUNT_IF(trim(`cod_material`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`cod_material` IS NOT NULL AND NOT trim(`cod_material`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`cod_material`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`cod_material`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'cod_fornecedor' AS coluna,
       COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`cod_fornecedor` IS NOT NULL AND NOT trim(`cod_fornecedor`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`cod_fornecedor`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`cod_fornecedor`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'num_contrato_basico' AS coluna,
       COUNT_IF(trim(`num_contrato_basico`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`num_contrato_basico` IS NOT NULL AND NOT trim(`num_contrato_basico`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`num_contrato_basico`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`num_contrato_basico`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'num_registro_info' AS coluna,
       COUNT_IF(trim(`num_registro_info`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`num_registro_info` IS NOT NULL AND NOT trim(`num_registro_info`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`num_registro_info`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`num_registro_info`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
UNION ALL
SELECT 'cod_requisicao_compra' AS coluna,
       COUNT_IF(trim(`cod_requisicao_compra`) RLIKE '^0[0-9]') AS com_zeros_esq,
       COUNT_IF(`cod_requisicao_compra` IS NOT NULL AND NOT trim(`cod_requisicao_compra`) RLIKE '^[0-9]+$') AS nao_numerico,
       COUNT(DISTINCT trim(`cod_requisicao_compra`)) AS distintos_string,
       COUNT(DISTINCT regexp_replace(trim(`cod_requisicao_compra`), '^0+', '')) AS distintos_sem_zeros
  FROM dev_procurement.corp_curated.tbl_ds_proc_me2n
ORDER BY com_zeros_esq DESC;

### 16.3 Escopo declarado da tabela

O comentário do `DESCRIBE` de `tp_documento_compras` admite: *"Na base atual o valor é
sempre F (pedido de compra) porque a CDS view de item não contém planos de entrega (categoria L)"*.

**Consequência:** planos de entrega **não existem** nesta tabela. Ao extrair a ME2N do SAP,
é preciso filtrar apenas categoria F — senão a comparação acusará falsos ausentes.

In [0]:
-- 16.3 ESCOPO: TIPO E CATEGORIA DO DOCUMENTO
SELECT tp_documento_compras, tp_categoria_documento,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
FROM base
GROUP BY tp_documento_compras, tp_categoria_documento
ORDER BY linhas DESC;

In [0]:
-- 16.3b CONSISTENCIA DO DE-PARA cod_categoria_item -> desc_categoria_item
SELECT cod_categoria_item,
       COUNT(DISTINCT desc_categoria_item) AS qtd_descricoes,
       CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(desc_categoria_item))) AS descricoes,
       COUNT(*) AS linhas,
       CASE WHEN COUNT(DISTINCT desc_categoria_item) > 1
            THEN 'ERRO - codigo com mais de uma descricao' ELSE 'ok' END AS veredito
FROM base
GROUP BY cod_categoria_item
ORDER BY qtd_descricoes DESC, linhas DESC;

### 16.4 Coerência das datas de remessa

In [0]:
-- 16.4 COERENCIA DAS DATAS DE REMESSA
SELECT 'remessa_primeira <= remessa_ultima' AS regra,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`)) IS NOT NULL AND COALESCE(to_date(CASE WHEN `dt_remessa_ultima` RLIKE '^[0-9]{8}$' THEN `dt_remessa_ultima` END, 'yyyyMMdd'), to_date(`dt_remessa_ultima`)) IS NOT NULL) AS avaliadas,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`)) > COALESCE(to_date(CASE WHEN `dt_remessa_ultima` RLIKE '^[0-9]{8}$' THEN `dt_remessa_ultima` END, 'yyyyMMdd'), to_date(`dt_remessa_ultima`))) AS violacoes
  FROM base
UNION ALL
SELECT 'pedido <= remessa_primeira' AS regra,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_pedido` RLIKE '^[0-9]{8}$' THEN `dt_pedido` END, 'yyyyMMdd'), to_date(`dt_pedido`)) IS NOT NULL AND COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`)) IS NOT NULL) AS avaliadas,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_pedido` RLIKE '^[0-9]{8}$' THEN `dt_pedido` END, 'yyyyMMdd'), to_date(`dt_pedido`)) > COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`))) AS violacoes
  FROM base
UNION ALL
SELECT 'criacao <= remessa_primeira' AS regra,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' THEN `dt_criacao` END, 'yyyyMMdd'), to_date(`dt_criacao`)) IS NOT NULL AND COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`)) IS NOT NULL) AS avaliadas,
       COUNT_IF(COALESCE(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' THEN `dt_criacao` END, 'yyyyMMdd'), to_date(`dt_criacao`)) > COALESCE(to_date(CASE WHEN `dt_remessa_primeira` RLIKE '^[0-9]{8}$' THEN `dt_remessa_primeira` END, 'yyyyMMdd'), to_date(`dt_remessa_primeira`))) AS violacoes
  FROM base
ORDER BY violacoes DESC;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- 90. INTEGRIDADE: cod_material -> dev_procurement.corp_curated.tbl_ds_mdm_mm60.cod_material
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM base WHERE `cod_material` IS NOT NULL AND trim(CAST(`cod_material` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '47', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       'num_pedido_compra, cod_material', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_pedido_compra + num_item_pedido_compra' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_pedido_compra + num_item_pedido_compra + cod_material' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `cod_material` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_pedido_compra' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_pedido_compra` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
